# Predictive Modeling

## Prepare my workspace

In [482]:
# Import needed libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sms

In [483]:
# Load the data
re_df = pd.read_csv(r"D:\Data science project\Data science\Codveda-internship\Level 1\Task 2\processed_real_estate.csv")

# preview the data
print("Real estates data:\n")
re_df

Real estates data:



,name,sqm,year_built,Price(EGP),Price/Sqm,has_main_facilities,has_security_features,has_parking_safety,has_building_facilities,has_other_features,...,propertySubType_Office,propertySubType_Others(very rare),propertySubType_Retail,log(sqm),log(price),log(price/sqm),Is log(price/sqm) outlier,Is log(sqm) outlier,Is log(price) outlier,Is FloorNumber outlier
0,Alamain (Latin District),92.86,2025,6594000,71010.1228,1,1,0,0,1,...,0,0,0,4.531093,15.701671,11.170578,False,False,False,False
1,Beachfront Tower - B1,398.00,2025,66104000,166090.4523,0,0,0,0,1,...,0,0,0,5.986452,18.006740,12.020288,False,False,True,False
2,PODIA,93.00,2025,12824000,137892.4731,0,1,1,0,1,...,1,0,0,4.532599,16.366829,11.834229,False,False,False,True
3,Mazarine Apartment,252.00,2025,12857000,51019.8413,1,1,0,0,0,...,0,0,0,5.529429,16.369399,10.839970,False,False,False,False
4,Alamain (Latin District),209.42,2025,8721000,41643.5870,1,1,0,0,1,...,0,0,0,5.344342,15.981244,10.636903,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
851,Alamain (Latin District),177.47,2025,5678000,31994.1399,1,1,0,0,1,...,0,0,0,5.178802,15.552110,10.373308,False,False,False,False
852,Jade Park,692.00,2025,25292000,36549.1329,1,0,1,1,0,...,0,0,0,6.539586,17.045999,10.506413,False,True,False,False
853,Mamsha Avenue,171.00,2025,6137000,35888.8889,1,1,0,0,1,...,0,0,0,5.141664,15.629847,10.488183,False,False,False,False
854,Alamain (Latin District),207.58,2025,5933000,28581.7516,1,1,0,0,1,...,0,0,0,5.335517,15.596041,10.260524,False,False,False,False


## Regression

### Pick the wanted features only

In [484]:
drop_cols = ["sqm","Price(EGP)","Price/Sqm",
             "year_built","FloorNumber","BuildingNumber",
             "UnitNumber","Is FloorNumber outlier","LowFloor",
             "Bedroom_0","Bathroom_0","city_Cairo",
             "governorate_Cairo","propertyCategory_Residential",
             "propertySubType_Apartment","log(price/sqm)","city_Mersa Matruh"] # The unnecessary columns, and first dummy from every categorical column of dummies for full rank dummies, Dropping log(price/sqm) because it has the dependent variable. Dropping governorate Matrouh because I have two values of Matrouh with different columns
reg_df = re_df.drop(drop_cols, axis=1) # drop these columns
outliers_bool = ["Is log(price) outlier", "Is log(price/sqm) outlier", "Is log(sqm) outlier"] # outliers boolean columns
outliers_mask = reg_df[outliers_bool].max(axis=1) # Split outliers from non-outliers
non_outliers_df = reg_df[~ outliers_mask].drop(outliers_bool, axis =1) # the non outliers rows
outliers_df = reg_df[outliers_mask].drop(outliers_bool, axis = 1) # the outlier values

non_outliers_df

,name,has_main_facilities,has_security_features,has_parking_safety,has_building_facilities,has_other_features,HighFloor,MidFloor,has_modern,has_facilities,...,governorate_Alexandria,governorate_Dakahlia,governorate_Matrouh,propertyCategory_Commercial,propertySubType_Attached Houses,propertySubType_Office,propertySubType_Others(very rare),propertySubType_Retail,log(sqm),log(price)
0,Alamain (Latin District),1,1,0,0,1,0,1,1,1,...,1,0,0,0,0,0,0,0,4.531093,15.701671
2,PODIA,0,1,1,0,1,1,0,0,0,...,0,0,0,1,0,1,0,0,4.532599,16.366829
3,Mazarine Apartment,1,1,0,0,0,0,1,0,1,...,0,0,1,0,0,0,0,0,5.529429,16.369399
4,Alamain (Latin District),1,1,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,5.344342,15.981244
5,Latin City,0,1,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,4.934474,15.715825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
849,T- Residences,1,1,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,4.867534,14.796937
850,Alamain (Latin District),1,1,0,0,1,0,1,1,1,...,0,0,1,0,0,0,0,0,4.835567,15.731256
851,Alamain (Latin District),1,1,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,5.178802,15.552110
853,Mamsha Avenue,1,1,0,0,1,0,1,0,1,...,0,0,0,0,0,0,0,0,5.141664,15.629847


### Non-outliers regression

In [485]:
# Set X, and y
x = non_outliers_df.drop(["name","log(price)"],axis= 1)
y = non_outliers_df["log(price)"]

def split_data(X,y, test_size:float = 0.2):
    """Split the data into train, and test data

    Args:
        X (DataFrame): X matrix
        y (array_like): The dependent variable
        test_size (float, optional): The percentage of test rows of the data. Defaults to 0.2.

    Returns:
        X_train, X_test, y_train, y_test: Return the train, and test data
    """
    # Split into train, and test data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2) # test data is 20% of the data which equals 146 row
    return [X_train, X_test, y_train, y_test]
X_train, X_test, y_train, y_test = split_data(x,y)

x

,has_main_facilities,has_security_features,has_parking_safety,has_building_facilities,has_other_features,HighFloor,MidFloor,has_modern,has_facilities,has_green_view,...,city_Old Cairo,governorate_Alexandria,governorate_Dakahlia,governorate_Matrouh,propertyCategory_Commercial,propertySubType_Attached Houses,propertySubType_Office,propertySubType_Others(very rare),propertySubType_Retail,log(sqm)
0,1,1,0,0,1,0,1,1,1,0,...,0,1,0,0,0,0,0,0,0,4.531093
2,0,1,1,0,1,1,0,0,0,1,...,0,0,0,0,1,0,1,0,0,4.532599
3,1,1,0,0,0,0,1,0,1,1,...,0,0,0,1,0,0,0,0,0,5.529429
4,1,1,0,0,1,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,5.344342
5,0,1,0,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,4.934474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
849,1,1,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,4.867534
850,1,1,0,0,1,0,1,1,1,0,...,0,0,0,1,0,0,0,0,0,4.835567
851,1,1,0,0,1,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,5.178802
853,1,1,0,0,1,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,5.141664


#### Make the original model

In [486]:
org_reg_model = LinearRegression().fit(X_train, y_train) # fit the model

def reg_summary(X_train, y_train):
    """Return the statsmodels model summary

    Args:
        X_train (DataFrame): X train matrix
        y_train (Array_like): Dependent variable
    Returns:
        Model_summary: Return the model summary(F-statistic, R-squared, etc.)
    """
### See the summary of the model using statsmodels

    reg_sms = sms.OLS(y_train, sms.add_constant(X_train)).fit()
    results = reg_sms.summary2()

    print(f"The regression model summary:\n{results}")
reg_summary(X_train,y_train)

The regression model summary:
                         Results: Ordinary least squares
Model:                   OLS                   Adj. R-squared:          0.805    
Dependent Variable:      log(price)            AIC:                     -91.4904 
Date:                    2025-10-28 21:32      BIC:                     48.4012  
No. Observations:        585                   Log-Likelihood:          77.745   
Df Model:                31                    F-statistic:             78.98    
Df Residuals:            553                   Prob (F-statistic):      4.63e-181
R-squared:               0.816                 Scale:                   0.047481 
---------------------------------------------------------------------------------
                                   Coef.  Std.Err.    t    P>|t|   [0.025  0.975]
---------------------------------------------------------------------------------
const                             10.7041   0.3418 31.3165 0.0000 10.0327 11.3755
has_main_fa

Model overall statistically strong:
- F-statistic: 77.83, p-value: 0.00 --> At least one of them effect on log(price)
- R-squared: 0.814, Adj. R-squared: 0.803 --> The model explained 80% of the data which is good
- AIC, BIC: -49.1556, 99.7359 both looks fine
- Skew: 0.90(roughly normal), Kurtosis: 2.832(Roughly normal)
-JB: 1.472, p-value: 0.479 --> Fail to reject normality of residuals
- Condition Number: 1 $\times$ $10^16$ Which is very high, indicating high multicollinearity, which return unreliable parameters, and values

> Solution: Drop one of governorate or city, Add the square of sqm because of non-linear relationship between them, Add interaction like has_modern, has main facilities, etc.

In [487]:
x = x.drop(["governorate_Matrouh","governorate_Alexandria","governorate_Dakahlia"],axis = 1)



X_train, X_test, y_train, y_test = split_data(x, y) # split the new X data
reg_summary(X_train,y_train) # see the new model summary

The regression model summary:
                         Results: Ordinary least squares
Model:                   OLS                   Adj. R-squared:          0.790    
Dependent Variable:      log(price)            AIC:                     -63.8705 
Date:                    2025-10-28 21:32      BIC:                     62.9062  
No. Observations:        585                   Log-Likelihood:          60.935   
Df Model:                28                    F-statistic:             79.41    
Df Residuals:            556                   Prob (F-statistic):      3.72e-174
R-squared:               0.800                 Scale:                   0.050019 
---------------------------------------------------------------------------------
                                   Coef.  Std.Err.    t    P>|t|   [0.025  0.975]
---------------------------------------------------------------------------------
const                             10.8491   0.3366 32.2327 0.0000 10.1880 11.5102
has_main_fa

That's better now CN: 369 which is very good

#### Model selection using General to specific method

In [488]:
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

def general_to_specific(df, y_col, x_cols, pval_threshold=0.05, vif_threshold=10.0,
                        correction='fdr_bh', verbose=True):
    """
    General-to-Specific model selection using AIC, corrected p-values, and VIF reduction.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Dataset containing dependent and independent variables.
    y_col : str
        Dependent variable name.
    x_cols : list
        Initial list of independent variable names.
    pval_threshold : float
        Threshold for corrected p-values (default = 0.05).
    vif_threshold : float
        Threshold for Variance Inflation Factor (default = 10).
    correction : str
        Type of p-value correction ('bonferroni', 'fdr_bh', etc.).
    verbose : bool
        Whether to print progress.

    Returns
    -------
    model : statsmodels.regression.linear_model.RegressionResultsWrapper
        Final OLS model.
    removed_vars : list
        Variables removed during the selection process.
    """
    removed_vars = []
    current_x = x_cols.copy()

    while True:
        X = sm.add_constant(df[current_x])
        y = df[y_col]
        model = sm.OLS(y, X).fit()

        # 1️⃣ Corrected p-values
        pvals = model.pvalues.drop('const', errors='ignore')
        _, corrected_pvals, _, _ = multipletests(pvals, method=correction)
        corrected = pd.Series(corrected_pvals, index=pvals.index)

        worst_p = corrected.max()
        worst_var_p = corrected.idxmax()

        # 2️⃣ VIF calculation
        vif_df = pd.DataFrame({
            'Variable': current_x,
            'VIF': [variance_inflation_factor(X.values, i+1) for i in range(len(current_x))]
        })
        worst_vif = vif_df['VIF'].max()
        worst_var_vif = vif_df.loc[vif_df['VIF'].idxmax(), 'Variable']

        # 3️⃣ Decide what to remove
        if worst_p > pval_threshold or worst_vif > vif_threshold:
            reason = None
            if worst_vif > vif_threshold and (worst_vif / 10) > (worst_p / pval_threshold):
                worst_var = worst_var_vif
                reason = f"VIF {worst_vif:.2f}"
            else:
                worst_var = worst_var_p
                reason = f"corrected p={worst_p:.4f}"

            old_aic = model.aic
            candidate_x = [v for v in current_x if v != worst_var]
            candidate_model = sm.OLS(y, sm.add_constant(df[candidate_x])).fit()

            if candidate_model.aic <= old_aic:
                if verbose:
                    print(f"Removing '{worst_var}' ({reason}, AIC improved {old_aic:.2f} → {candidate_model.aic:.2f})")
                current_x = candidate_x
                removed_vars.append(worst_var)
                model = candidate_model
            else:
                if verbose:
                    print(f"Stopped: Removing '{worst_var}' worsens AIC ({old_aic:.2f} → {candidate_model.aic:.2f})")
                break
        else:
            if verbose:
                print("No variable exceeds corrected p-value or VIF threshold. Selection complete.")
            break

    # Final VIFs for sanity check
    final_X = sm.add_constant(df[current_x])
    final_vif = pd.DataFrame({
        'Variable': current_x,
        'VIF': [variance_inflation_factor(final_X.values, i+1) for i in range(len(current_x))]
    })

    if verbose:
        print("\nFinal VIFs:")
        print(final_vif.sort_values('VIF', ascending=False).to_string(index=False))

    return model, removed_vars, final_vif


df = X_train.join(y_train)
final_model, removed_vars, final_vif = general_to_specific(df, "log(price)", X_train.columns)
X_train = X_train.drop(removed_vars, axis = 1)
print(f"The final model summary:\n{final_model.summary2()}")

Removing 'propertyCategory_Commercial' (VIF inf, AIC improved -63.87 → -63.87)
Removing 'has_other_features' (corrected p=0.9768, AIC improved -63.87 → -65.86)
Removing 'Bathroom_2' (corrected p=0.9839, AIC improved -65.86 → -67.86)
Removing 'propertySubType_Others(very rare)' (corrected p=0.9508, AIC improved -67.86 → -69.85)
Removing 'has_security_features' (corrected p=0.6474, AIC improved -69.85 → -71.64)
Removing 'has_building_facilities' (corrected p=0.3054, AIC improved -71.64 → -72.54)
Removing 'has_green_view' (corrected p=0.5476, AIC improved -72.54 → -74.16)
Removing 'city_North Coast' (corrected p=0.3592, AIC improved -74.16 → -75.28)
Removing 'has_facilities' (corrected p=0.1957, AIC improved -75.28 → -75.54)
Stopped: Removing 'Bedroom_3' worsens AIC (-75.54 → -42.46)

Final VIFs:
                       Variable       VIF
                      Bedroom_3 28.375303
                      Bedroom_2 16.256948
                Bedroom_above 3 15.671140
                      Bedro

In [489]:
def vifs(X_train):
    """Return the VIFS values

    Args:
        X_train (dataframe): X matrix
    """
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    vif_data = pd.DataFrame()
    vif_data["Variable"] = X_train.columns
    vif_data["VIF"] = [variance_inflation_factor(X_train.values, i) for i in range(X_train.shape[1])]
    print(vif_data)
vifs(X_train)

                           Variable         VIF
0               has_main_facilities    7.928716
1                has_parking_safety    1.646981
2                         HighFloor    1.701566
3                          MidFloor    2.585448
4                        has_modern    5.652472
5                 has_water_feature   10.165624
6                         Bedroom_1   11.041741
7                         Bedroom_2   17.078167
8                         Bedroom_3   50.973815
9                   Bedroom_above 3   18.301137
10                       Bathroom_1    1.437515
11                 Bathroom_above 2    2.133213
12                    city_Mansoura   11.998358
13  city_New Administrative Capital    4.421500
14                   city_New Cairo    1.410558
15                   city_Old Cairo    3.631041
16  propertySubType_Attached Houses    9.026524
17           propertySubType_Office    5.047407
18           propertySubType_Retail    4.024984
19                         log(sqm)  131

- There is strong multicollinearity inlog(sqm)

> Solutions center log(sqm)

In [490]:
X_train["log(sqm)_centered"] = X_train["log(sqm)"] - X_train["log(sqm)"].mean()


X_train = X_train.drop(["log(sqm)"], axis =1)

In [491]:
final_model = LinearRegression().fit(X_train, y_train)



- That's better, I should do the RESET to check the linearity, and residuals, and y_pred plot

In [492]:
from statsmodels.stats.diagnostic import linear_reset
import matplotlib.pyplot as plt
import seaborn as sns

reset_test_results = linear_reset(final_model, power=2, use_f = True) # RESET test
print(f"F-stat: {reset_test_results.fvalue:.3f}, p-value: {reset_test_results.pvalue:.4f}")

### Store y_pred, and residuals to plot the scatterplot
y_pred = final_model.fittedvalues
residuals = final_model.resid

sns.scatterplot(x=y_pred, y=residuals);
plt.title("Linearity Check")
plt.xlabel("Fitted values")
plt.ylabel("Residuals");

TypeError: result must come from a linear regression model

Fail to reject the normality of residuals, model is well-specified. Linearity assumptions like residuals are random aren't violated

#### Model metrics